# Baseline — Molecular Energy Prediction

**Competition:** predict the total energy (eV) of small molecules. `structures.csv`
lists every atom of every molecule (element, 3-D coordinates, Mulliken charge);
`train.csv` gives the target energy for 700 molecules.

- **Task:** regression — predict `target_energy_eV` per molecule
- **Metric:** (see competition page — lower error is better)
- **Kaggle link:** _TODO: add link_

**Approach:** molecular energy is dominated by *composition* — which atoms the molecule
contains. We aggregate per-molecule features (atom counts per element, charge and
geometry statistics) and fit a gradient-boosted-trees model.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
structures = pd.read_csv(f"{DATA_DIR}/structures.csv")
print(train.shape, test.shape, structures.shape)
structures.head(3)

(700, 2) (300, 1) (22193, 8)


,id,atom_index,atom,atomic_number,x,y,z,mulliken_charge
0,mol_000000,0,C,6,-0.018110,1.490623,0.010180,-0.517161
1,mol_000000,1,C,6,0.001541,0.038700,0.002304,0.215557
2,mol_000000,2,C,6,0.017925,-1.171625,-0.004285,-0.182591


In [2]:
# Per-molecule features: element counts + charge/geometry aggregates
counts = structures.pivot_table(index="id", columns="atom", aggfunc="size", fill_value=0)
counts.columns = [f"n_{c}" for c in counts.columns]

agg = structures.groupby("id").agg(
    n_atoms=("atom_index", "size"),
    z_sum=("atomic_number", "sum"),
    z_mean=("atomic_number", "mean"),
    charge_mean=("mulliken_charge", "mean"),
    charge_std=("mulliken_charge", "std"),
    x_range=("x", lambda s: s.max() - s.min()),
    y_range=("y", lambda s: s.max() - s.min()),
    z_range=("z", lambda s: s.max() - s.min()),
)
feats = counts.join(agg).fillna(0)
X  = feats.loc[train["id"]].values
Xt = feats.loc[test["id"]].values
y  = train["target_energy_eV"].values
print(X.shape, Xt.shape)

(700, 11) (300, 11)


In [3]:
# 5-fold CV
oof = np.zeros(len(train)); preds = np.zeros(len(test))
for tr_idx, va_idx in KFold(5, shuffle=True, random_state=0).split(X):
    m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, random_state=0, verbose=-1)
    m.fit(X[tr_idx], y[tr_idx])
    oof[va_idx] = m.predict(X[va_idx])
    preds += m.predict(Xt) / 5
print(f"CV MAE: {mean_absolute_error(y, oof):.2f} eV")

CV MAE: 66.02 eV


In [4]:
sub = pd.DataFrame({"id": test["id"], "target_energy_eV": preds})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,target_energy_eV
0,mol_000265,-10553.721336
1,mol_000781,-11614.436418
2,mol_000186,-9567.129336
3,mol_000185,-8445.581626
4,mol_000029,-9525.793556


## Ideas to improve

- A near-perfect linear feature exists: energy is almost additive per atom — fit a
  **linear regression on element counts alone** and compare.
- Add pairwise-distance features (bond counts by element pair within a distance cutoff).
- Graph neural networks (SchNet, DimeNet) are the state of the art for this task.
